### PREPROCESAMIENTO BBI / BBII


In [1]:
import matplotlib.pyplot as plt


In [2]:
import xarray as xr
import regionmask
from shapely.geometry import Polygon
import os
import geopandas as gpd

In [3]:
import os

In [10]:
import pandas as pd

1️⃣ Rutas y carpetas

## pH

In [12]:
# Definir paths relativos
BASE_DIR = os.path.abspath("..")  # un nivel arriba (repo root)
DATA_RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

# Nombre del archivo NetCDF que ya bajaste
ph_file = os.path.join(DATA_RAW_DIR, "cmems_obs-mob_glo_bgc-car_my_irr-i_1756152669355.nc")

# Verificar que existe
if not os.path.isfile(ph_file):
    raise FileNotFoundError(f"No se encuentra el archivo: {ph_file}\nColocalo en data/raw/")

# Abrir NetCDF con xarray
ds_ph = xr.open_dataset(ph_file)
print(ds_ph)

<xarray.Dataset> Size: 12MB
Dimensions:               (time: 468, latitude: 14, longitude: 37)
Coordinates:
  * time                  (time) datetime64[ns] 4kB 1985-01-01 ... 2023-12-01
  * latitude              (latitude) float32 56B -55.88 -55.62 ... -52.88 -52.62
  * longitude             (longitude) float32 148B -63.12 -62.88 ... -54.12
Data variables:
    omega_ar              (time, latitude, longitude) float32 970kB ...
    omega_ar_uncertainty  (time, latitude, longitude) float32 970kB ...
    omega_ca              (time, latitude, longitude) float32 970kB ...
    omega_ca_uncertainty  (time, latitude, longitude) float32 970kB ...
    ph                    (time, latitude, longitude) float32 970kB ...
    ph_uncertainty        (time, latitude, longitude) float32 970kB ...
    spco2                 (time, latitude, longitude) float32 970kB ...
    spco2_uncertainty     (time, latitude, longitude) float32 970kB ...
    talk                  (time, latitude, longitude) float32 970

2️⃣ Abrir dataset completo

In [3]:
ds = xr.open_dataset(data_raw_ph)

## Oxigeno

El hindcast

In [5]:
# Definir paths relativos
BASE_DIR = os.path.abspath("..")  # un nivel arriba (repo root)
DATA_RAW_DIR = os.path.join(BASE_DIR, "data", "raw", "Oxigeno")
DATA_PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

# Nombre del archivo NetCDF del oxígeno (ajustá el nombre exacto si difiere)
o2_file = os.path.join(DATA_RAW_DIR, "cmems_mod_glo_bgc_my_0.25deg_P1M-m_1757960017296.nc")

# Verificar que existe
if not os.path.isfile(o2_file):
    raise FileNotFoundError(f"No se encuentra el archivo: {o2_file}\nColocalo en data/raw/Oxigeno/")

# Abrir NetCDF con xarray
ds_o2 = xr.open_dataset(o2_file)
print(ds_o2)

<xarray.Dataset> Size: 444MB
Dimensions:    (time: 360, depth: 66, latitude: 14, longitude: 37)
Coordinates:
  * time       (time) datetime64[ns] 3kB 1993-01-01 1993-02-01 ... 2022-12-01
  * depth      (depth) float32 264B 0.5058 1.556 2.668 ... 3.898e+03 4.093e+03
  * latitude   (latitude) float32 56B -56.0 -55.75 -55.5 ... -53.25 -53.0 -52.75
  * longitude  (longitude) float32 148B -63.0 -62.75 -62.5 ... -54.25 -54.0
Data variables:
    chl        (time, depth, latitude, longitude) float32 49MB ...
    fe         (time, depth, latitude, longitude) float32 49MB ...
    no3        (time, depth, latitude, longitude) float32 49MB ...
    nppv       (time, depth, latitude, longitude) float32 49MB ...
    o2         (time, depth, latitude, longitude) float32 49MB ...
    ph         (time, depth, latitude, longitude) float32 49MB ...
    phyc       (time, depth, latitude, longitude) float32 49MB ...
    po4        (time, depth, latitude, longitude) float32 49MB ...
    si         (time, dep

el Glodap

In [12]:
glodap_file = os.path.join(DATA_RAW_DIR, "cmems_obs-ins_glo_bgc-car_my_glodap-obs_irr_1757964471340.csv")

# Mostrar las primeras líneas (ver si hay encabezado o metadatos)
with open(glodap_file, 'r', encoding='utf-8', errors='ignore') as f:
    for i in range(15):
        print(f.readline().strip())

# Credits: EU Copernicus Marine Service information - https://marine.copernicus.eu
# Product: INSITU_GLO_BGC_CARBON_DISCRETE_MY_013_050
# Dataset: cmems_obs-ins_glo_bgc-car_my_glodap-obs_irr
# Variable: Sea water alkalinity per unit mass (ALKW) µmol kg-1

parameter,platformId,platformType,time,longitude,latitude,depth,pressure,value,valueQc
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,3930,4003.1,204.28,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,2462,2499.1,185.83,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,2952,3000.4,193.62,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,3443,3503,200.39,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,3443,3503,200.39,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,1231,1246.2,174.15,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,1478,1497.1,168.72,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,1717,1739.6,171.28,1
DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,1971,19

In [13]:
glodap_file = os.path.join(DATA_RAW_DIR, "cmems_obs-ins_glo_bgc-car_my_glodap-obs_irr_1757964471340.csv")

# Leer el CSV saltando los metadatos
df_glodap = pd.read_csv(glodap_file, comment='#')

# Verificar que se cargó correctamente
print(df_glodap.head())
print(df_glodap.columns)

  parameter platformId platformType                      time  longitude  \
0      DOX2       ZDLP           BO  1995-03-23T20:18:00.000Z    -58.611   
1      DOX2       ZDLP           BO  1995-03-23T20:18:00.000Z    -58.611   
2      DOX2       ZDLP           BO  1995-03-23T20:18:00.000Z    -58.611   
3      DOX2       ZDLP           BO  1995-03-23T20:18:00.000Z    -58.611   
4      DOX2       ZDLP           BO  1995-03-23T20:18:00.000Z    -58.611   

   latitude   depth  pressure   value  valueQc  
0   -55.514  3930.0    4003.1  204.28        1  
1   -55.514  2462.0    2499.1  185.83        1  
2   -55.514  2952.0    3000.4  193.62        1  
3   -55.514  3443.0    3503.0  200.39        1  
4   -55.514  3443.0    3503.0  200.39        1  
Index(['parameter', 'platformId', 'platformType', 'time', 'longitude',
       'latitude', 'depth', 'pressure', 'value', 'valueQc'],
      dtype='object')


In [16]:
df_glodap_o2 = df_glodap[df_glodap['parameter'] == 'DOX2']

df_glodap_o2

,parameter,platformId,platformType,time,longitude,latitude,depth,pressure,value,valueQc
0,DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,3930.00,4003.100,204.280,1
1,DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,2462.00,2499.100,185.830,1
2,DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,2952.00,3000.400,193.620,1
3,DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,3443.00,3503.000,200.390,1
4,DOX2,ZDLP,BO,1995-03-23T20:18:00.000Z,-58.611,-55.514,3443.00,3503.000,200.390,1
...,...,...,...,...,...,...,...,...,...,...
992,DOX2,EBBW,BO,2010-02-09T00:00:00.000Z,-61.667,-53.298,98.50,99.284,294.522,8
993,DOX2,EBBW,BO,2010-02-09T00:00:00.000Z,-61.667,-53.298,121.75,122.685,293.652,8
994,DOX2,EBBW,BO,2010-02-09T00:00:00.000Z,-61.667,-53.298,48.75,49.167,292.818,8
995,DOX2,EBBW,BO,2010-02-09T00:00:00.000Z,-61.667,-53.298,10.00,9.937,296.018,8


In [18]:
# --- Definir paths relativos ---
BASE_DIR = os.path.abspath("..")  # un nivel arriba (repo root)
DATA_RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
SST_DIR = os.path.join(DATA_RAW_DIR, "SST")

# --- Archivos NetCDF que descargaste ---
hist_file = os.path.join(SST_DIR, "cmems_mod_glo_phy_my_0.083deg_P1M-m_1716909393502.nc")
nrt_file  = os.path.join(SST_DIR, "cmems_mod_glo_phy_myint_0.083deg_P1M-m_1759777081321.nc")

# Verificar que existen
for f in [hist_file, nrt_file]:
    if not os.path.isfile(f):
        raise FileNotFoundError(f"No se encuentra el archivo: {f}")

# --- Abrir NetCDF ---
ds_hist = xr.open_dataset(hist_file)
ds_nrt = xr.open_dataset(nrt_file)

# --- Revisar variables ---
print(ds_hist)
print(ds_nrt)

# --- Seleccionar solo SST superficial ---
# normalmente la variable es 'thetao' y la primera profundidad es superficial
# revisar primero si tiene dimensiones de profundidad
if 'depth' in ds_hist.dims:
    ds_hist_surf = ds_hist['thetao'].isel(depth=0)
    ds_nrt_surf  = ds_nrt['thetao'].isel(depth=0)
else:
    ds_hist_surf = ds_hist['thetao']
    ds_nrt_surf  = ds_nrt['thetao']

# --- Concatenar por tiempo ---
ds_sst = xr.concat([ds_hist_surf, ds_nrt_surf], dim='time')

# --- Guardar archivo concatenado ---
output_file = os.path.join(SST_DIR, "sst_combined.nc")
ds_sst.to_netcdf(output_file)

print(f"Archivo concatenado guardado en: {output_file}")

<xarray.Dataset> Size: 19MB
Dimensions:    (time: 342, latitude: 42, longitude: 110, depth: 1)
Coordinates:
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 168B -56.08 -56.0 -55.92 ... -52.75 -52.67
  * longitude  (longitude) float32 440B -63.17 -63.08 -63.0 ... -54.17 -54.08
  * time       (time) datetime64[ns] 3kB 1993-01-01 1993-02-01 ... 2021-06-01
Data variables:
    bottomT    (time, latitude, longitude) float32 6MB ...
    so         (time, depth, latitude, longitude) float32 6MB ...
    thetao     (time, depth, latitude, longitude) float32 6MB ...
Attributes:
    Conventions:       CF-1.11
    title:             Monthly mean fields for product GLOBAL_REANALYSIS_PHY_...
    institution:       Mercator Ocean
    producer:          CMEMS - Global Monitoring and Forecasting Centre
    source:            MERCATOR GLORYS12V1
    credit:            E.U. Copernicus Marine Service Information (CMEMS)
    contact:           servicedesk.cmems@mercator-ocean.eu
  